In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

# 12 - Merge RegBL and Area Statistics

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `15_OFS_area_statistics`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Area statistics

In [ ]:
gdf_stats = gpd.read_file(f"{ROOT}data/raw/15_OFS_area_statistics/arealstatistik_2056.gpkg")

gdf_stats.head()

In [ ]:
list(gdf_stats.columns)

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.8.csv', index_col=0)

df_regbl.head()

## Convert current dataset into GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Perform nearest spatial join

In [ ]:
# 1) Reset the index of df_regbl and store it as a column named "orig_idx" 
#    to preserve the original index for tracking and restoring later.
df_tmp = df_regbl.reset_index().rename(columns={"index": "orig_idx"})

# 2) Perform a nearest spatial join between df_tmp and gdf_stats.
#    This adds columns from gdf_stats to df_tmp based on the nearest geometry,
#    and calculates the distance to the nearest feature in gdf_stats.
nearest = gpd.sjoin_nearest(
    df_tmp,
    gdf_stats,
    how="left",
    distance_col="distance_to_stats"
)

# 3) For each "orig_idx", keep only the single closest match by:
#    - Sorting by "distance_to_stats" to ensure the nearest match is first.
#    - Dropping duplicate rows for each "orig_idx", keeping the first (nearest) match.
nearest = (
    nearest
    .sort_values("distance_to_stats")  # Sort by distance (nearest first)
    .drop_duplicates(subset="orig_idx", keep="first")  # Keep only the closest match
)

# 4) Restore the original index by:
#    - Setting "orig_idx" back as the index.
#    - Reordering rows to match the original order of df_tmp.
joined_gdf = (
    nearest
    .set_index("orig_idx")  # Restore original index
    .loc[df_tmp["orig_idx"]]  # Reorder rows to match the original order
)

# At this point, joined_gdf:
#   - Has the same number of rows as df_regbl.
#   - Retains the same index and ordering as df_regbl.


In [ ]:
joined_gdf.shape

In [ ]:
joined_gdf.head()

In [ ]:
df_regbl = joined_gdf

## Rename columns

In [ ]:
cols = [
 'RELI',
 'E_COORD',
 'N_COORD',
 'GMDE_ID',
 'GMDE_HISTID',
 'FJ85',
 'FJ97',
 'FJ09',
 'FJ18',
 'FJ25',
 'METHOD25',
 'REVISION25',
 'AS85_72',
 'AS97_72',
 'AS09_72',
 'AS18_72',
 'AS25_72',
 'AS85_27',
 'AS97_27',
 'AS09_27',
 'AS18_27',
 'AS25_27',
 'AS85_17',
 'AS97_17',
 'AS09_17',
 'AS18_17',
 'AS25_17',
 'AS85_4',
 'AS97_4',
 'AS09_4',
 'AS18_4',
 'AS25_4',
 'LC85_27',
 'LC97_27',
 'LC09_27',
 'LC18_27',
 'LC25_27',
 'LC85_6',
 'LC97_6',
 'LC09_6',
 'LC18_6',
 'LC25_6',
 'LU85_46',
 'LU97_46',
 'LU09_46',
 'LU18_46',
 'LU25_46',
 'LU85_10',
 'LU97_10',
 'LU09_10',
 'LU18_10',
 'LU25_10',
 'LU85_4',
 'LU97_4',
 'LU09_4',
 'LU18_4',
 'LU25_4',
 'geometry'
 ]

# Add 'ArealStat - ' to the beginning of each column name except for 'geometry'
for col in cols:
    if col != 'geometry':
        df_regbl.rename(columns={col: 'ArealStat - ' + col}, inplace=True)

## Convert back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.9.csv")